In [3]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used
# Energy emission peaks in MeV
K40_MEV = [1.460]
TL208_MEV = [2.614]
CS137_MEV = [0.661]
ANNIHIL_MEV = 0.511
NA22_MEV = [ANNIHIL_MEV, 1.2745, 1.2745+ANNIHIL_MEV]
CO60_MEV = [(1.1732+1.3325)/2, 1.1732+1.3325] # First value is the average of the two photopeaks. They have the same branching ratio

colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kViolet, ROOT.kYellow, ROOT.kSpring, ROOT.kCyan,]

In [4]:
files_A = sorted((project_root/'data/260519').glob('SUBT_A0[1-4]*.csv'))
files_B = sorted((project_root/'data/260519').glob('SUBT_B0[1-4]*.csv'))
print(files_A)
print(files_B)


[PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_A01_NewSourceTest_Background.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_A02_NewSourceTest_Cs137.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_A03_NewSourceTest_Na22.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_A04_NewSourceTest_Co60.csv')]
[PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B01_NewSourceTest_Background.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B02_NewSourceTest_Cs137.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B03_NewSourceTest_Na22.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B04_NewSourceTest_Co60.csv')]


In [5]:
acqs_A = [SiphraAcquisition(file) for file in files_A]
acqs_B = [SiphraAcquisition(file) for file in files_B]
print(acqs_A)
print(acqs_B)

[SIPHRAAcquisition(File: 'SUBT_A01_NewSourceTest_Background'), SIPHRAAcquisition(File: 'SUBT_A02_NewSourceTest_Cs137'), SIPHRAAcquisition(File: 'SUBT_A03_NewSourceTest_Na22'), SIPHRAAcquisition(File: 'SUBT_A04_NewSourceTest_Co60')]
[SIPHRAAcquisition(File: 'SUBT_B01_NewSourceTest_Background'), SIPHRAAcquisition(File: 'SUBT_B02_NewSourceTest_Cs137'), SIPHRAAcquisition(File: 'SUBT_B03_NewSourceTest_Na22'), SIPHRAAcquisition(File: 'SUBT_B04_NewSourceTest_Co60')]


In [6]:
[*files_A, *files_B]

[PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_A01_NewSourceTest_Background.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_A02_NewSourceTest_Cs137.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_A03_NewSourceTest_Na22.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_A04_NewSourceTest_Co60.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B01_NewSourceTest_Background.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B02_NewSourceTest_Cs137.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B03_NewSourceTest_Na22.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260519/SUBT_B04_NewSourceTest_Co60.csv')]

In [7]:
# histograms
from collections import namedtuple

DataHandler = namedtuple('DataHandler', ['acq', 'hist', 'src', 'subt'])

hists = {}
sources = []
handlers = {}
for acq, crystal in zip([*acqs_A, *acqs_B], [*('A'*4), *('B'*4)]):
    filepath = acq.filepath.stem
    src = (MetadataLoader.load(acq.metadataFile)).source
    src_key = 'BG' if src == 'Background' else src[:2]
    sources.append(src)
    siphra_code = acq.filepath.stem[5]
    print(src)
    # Generate histograms
    hist = ROOT.TH1F(f"{src}_SIPHRA_{siphra_code}", "", BITS12, 0, BITS12)
    hist.Fill(acq['s']/len(acq.active_chs))
    hists[f'{crystal}_{src_key}'] = hist
    handlers[f'{crystal}_{src_key}'] = DataHandler(acq, hist, src, None)
    del hist

Background
Cs-137
Na-22
Co-60
Background
Cs-137
Na-22
Co-60


In [38]:
# Subtract backgrounds here if correction factors are not configured, checking each one

correction_factors = {
    'A_Cs': 0.7,
    'A_Na': 1.05,
    'A_Co': 1.2,
    'B_Cs': 1,
    'B_Na': 1.1,
    'B_Co': 1.25,
}

if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

Yinit = 0.82 # For stat boxes

cv = ROOT.TCanvas('cv', 'cv', 800, 600)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgd = ROOT.TLegend(0.48, 0.61, 0.75, 0.83)

whichSiphra = 'B'
source = 'Co'
bg = handlers[f'{whichSiphra}_BG']
data_key = f'{whichSiphra}_{source}'
sgnl = handlers[data_key]

bg_scaled = bg.hist.Clone()
correction_factor = correction_factors[data_key] if correction_factors[data_key] else 1
bg_scaled.Scale(correction_factor*sgnl.acq.exposure/bg.acq.exposure)

subtracted = sgnl.hist.Clone(f'{sgnl.src}_{whichSiphra}_Subt.')
subtracted.Add(bg_scaled, -1)
# subts[data_key].hist = subtracted

lgd.AddEntry(sgnl.hist, 'Signal')
lgd.AddEntry(bg_scaled, 'Background')
lgd.AddEntry(subtracted, 'Subtracted')
sgnl.hist.SetLineColor(colors[0])
bg_scaled.SetLineColor(colors[1])
subtracted.SetLineColor(colors[2])
sgnl.hist.SetTitle(f'{sgnl.src} SIPHRA {data_key[:1]}')
sgnl.hist.Draw('hist')
bg_scaled.Draw('hist sames')
subtracted.Draw('hist sames')
cv.SetLogy()
lgd.Draw()
cv.Draw()

new_handler = DataHandler(sgnl.acq, sgnl.hist, sgnl.src, subtracted)
handlers[data_key] = new_handler

In [98]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

Yinit = 0.82 # For stat boxes

cv = ROOT.TCanvas('cv', 'cv', 1600, 600)
cv.Divide(2,1)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(2)]

srcs_list = list(handlers.values())
srcs_list.pop(4)
srcs_list.pop(0)


for n, handler in enumerate(srcs_list):
    cv.cd(n//3 + 1)
    hist = handler.subt
    lgds[n//3].AddEntry(hist, handler.src)
    if not n%3:
        s = 'A' if n < 3 else 'B'
        hist.SetTitle(f'SIPHRA {s} subtracted spectra')
        hist.SetLineColor(colors[n%3+3]+2)
        hist.Draw('hist')
    else:
        hist.SetLineColor(colors[n%3+3]+2)
        hist.Draw('hist sames')

for c in range(2):
    cv.cd(c+1)
    lgds[c].Draw()
    cv.cd(c+1).SetLogy()

cv.Draw()

    

In [48]:
srcs_list = list(handlers.values())
srcs_list.pop(4)
srcs_list.pop(0)
srcs_list

[DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_A02_NewSourceTest_Cs137'), hist=<cppyy.gbl.TH1F object at 0x5eea26f0fe90>, src='Cs-137', subt=<cppyy.gbl.TH1F object at 0x5eea29aa4c10>),
 DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_A03_NewSourceTest_Na22'), hist=<cppyy.gbl.TH1F object at 0x5eea21ddc690>, src='Na-22', subt=<cppyy.gbl.TH1F object at 0x5eea29bc73f0>),
 DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_A04_NewSourceTest_Co60'), hist=<cppyy.gbl.TH1F object at 0x5eea21d491f0>, src='Co-60', subt=<cppyy.gbl.TH1F object at 0x5eea29be4240>),
 DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_B02_NewSourceTest_Cs137'), hist=<cppyy.gbl.TH1F object at 0x5eea26342780>, src='Cs-137', subt=<cppyy.gbl.TH1F object at 0x5eea29a51210>),
 DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_B03_NewSourceTest_Na22'), hist=<cppyy.gbl.TH1F object at 0x5eea270305d0>, src='Na-22', subt=<cppyy.gbl.TH1F object at 0x5eea298999f0>),
 DataHandler(acq=SIPHRAAcquisition(File: 'SUBT_B04_NewSourceTest_Co60'), hist=<cp

In [94]:
# Fit ranges for each peak. Same order as in srcs_list
fit_ranges = [
    [(81,201)], # Cs siphra A
    [(89,152),(254,349),(380,431)], # Na siphra A
    [(228,351),(503,587)], # Co siphra A
    [(88,2011)],
    [(85,167),(261,374),(406,477)],
    [(240,376),(525, 612)]
]



4
9


In [104]:
mydict = {'a':1, 'b':2, 'c':3, 'd':4}
mydc = mydict.copy()
mydc['a']=50
mydict

{'a': 1, 'b': 2, 'c': 3, 'd': 4}

In [105]:
mydc

{'a': 50, 'b': 2, 'c': 3, 'd': 4}